# Site Interaction Analysis: launch-month corrected

This notebook fixes the month-indexing bug in the original analysis.

The key rule is now simple:
- launch month = the month that contains `operational_start_date`
- the month before launch = 0
- earlier months are negative
- all pre/post windows are anchored to the **launch month**, not the exact launch day and not the panel's global January 2024 calendar

That means if a site starts on `2024-04-12`, then `2024-04` is month 1 for that site.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

DATA_DIR = Path('/Users/dhruvsood/sonnysDataCollection/hypothesis-testing')
OUT_DIR = DATA_DIR / 'interaction_outputs'
OUT_DIR.mkdir(exist_ok=True)

sys.path.append(str(DATA_DIR))

from site_interaction_analysis_lib import (
    METRICS,
    build_pair_deltas,
    build_pair_event_profile,
    build_panel,
    build_sites,
    build_triple_deltas,
    build_triple_event_profile,
    choose_pair_examples,
    choose_triple_examples,
    configure_plotting,
    curate_outputs,
    find_pairs,
    find_triples,
    fmt_pct,
    plot_pair_distance_bands,
    plot_pair_event_profile,
    plot_pair_examples,
    plot_pair_examples_all,
    plot_pair_examples_existing_single_new_multi,
    plot_pair_pattern_existing_single_new_multi,
    plot_pair_summary,
    plot_triple_event_profile,
    plot_triple_examples_all,
    plot_triple_networks,
    plot_triple_summary,
    summarize_pairs,
    summarize_triples,
    write_report,
)

configure_plotting()
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

MAX_NEIGHBOR_MILES = 10.0
PRE_POST_WINDOW = 6
print('Writing artefacts to', OUT_DIR)


## 1. Build the monthly panel and correct the site-relative month number

The original `less_than-2yrs.csv` month numbering was tied to the panel calendar, which made some sites start at month 1 **before** they were operational.

Here we recompute site age from scratch using `operational_start_date`.


In [ ]:
panel, validation = build_panel(DATA_DIR)

print(f"Unified panel: {validation['sites']} sites, {validation['rows']:,} site-month rows")
print(f"Panel coverage: {validation['month_min'].date()} to {validation['month_max'].date()}")
print(f"lt2 rows whose month number changed after the fix: {validation['lt2_rows_with_changed_month_number']:,}")
print(f"lt2 sites affected: {validation['lt2_sites_with_changed_month_number']}")
print(f"Raw lt2 prelaunch rows still present in source data: {validation['lt2_prelaunch_rows']:,}")

display(Markdown(
    f"**Why this matters**\n\n"
    f"- We found **{validation['lt2_rows_with_changed_month_number']:,}** rows where the old `month_number` did not match the launch-month logic.\n"
    f"- Those rows span **{validation['lt2_sites_with_changed_month_number']}** different newer sites.\n"
    f"- From now on, the notebook uses `site_month_number`, not the raw `month_number` from the CSV."
))

validation['examples']


## 2. Build the site table and identify nearby launch pairs

We still define the local market using straight-line distance, with a same-ZIP preference when a same-ZIP option exists inside the radius.


In [ ]:
sites, site_lookup, site_distances = build_sites(panel)
pairs_df = find_pairs(sites, site_distances, max_neighbor_miles=MAX_NEIGHBOR_MILES, pre_buffer_months=PRE_POST_WINDOW)

print(f"Sites with coordinates: {len(sites)}")
print(f"Sites with a real launch month: {int(sites['has_launch_month'].sum())}")
print(f"Two-body candidate pairs inside {MAX_NEIGHBOR_MILES:.0f} miles: {len(pairs_df)}")
print(f"Same-ZIP candidate pairs selected: {int(pairs_df['zip_match'].sum())}")

pairs_df.head(10)


## 3. Two-body results

The event month is the new site's **launch month**, and the existing site is compared across the 6 months before vs the 6 months after that launch month.


In [ ]:
pair_deltas = build_pair_deltas(panel, pairs_df, pre_post_window=PRE_POST_WINDOW, min_months=3)
pair_summary = summarize_pairs(pair_deltas)
pair_profile = build_pair_event_profile(pair_deltas, panel, window=PRE_POST_WINDOW)
pair_examples = choose_pair_examples(pair_deltas, n_examples=4)

pair_deltas.to_csv(OUT_DIR / 'two_body_pair_deltas.csv', index=False)
pair_summary.to_csv(OUT_DIR / 'two_body_summary.csv', index=False)
pair_profile.to_csv(OUT_DIR / 'two_body_event_profile.csv', index=False)

print(f"Usable two-body pairs after the coverage filter: {len(pair_deltas)}")
display(Markdown(
    f"**Headline read**\n\n"
    f"- Median existing-site total change: **{fmt_pct(pair_deltas['existing_pct_wash_count_total'].median())}**\n"
    f"- Median combined-market total change: **{fmt_pct(pair_deltas['combined_pct_wash_count_total'].median())}**\n"
    f"- Same-ZIP share among usable pairs: **{pair_deltas['zip_match'].mean() * 100:.0f}%**"
))

pair_summary


In [ ]:
plot_pair_summary(pair_deltas, OUT_DIR / 'two_body_summary_ranges.png')
plot_pair_event_profile(pair_profile, OUT_DIR / 'two_body_event_time_total.png')
plot_pair_distance_bands(pair_deltas, OUT_DIR / 'two_body_distance_bands_total.png')
plot_pair_examples(pair_examples, panel, OUT_DIR / 'two_body_examples_total.png', window=PRE_POST_WINDOW)
plot_pair_examples_all(pair_deltas, panel, OUT_DIR / 'two_body_examples_all_sites.png')
plot_pair_examples_existing_single_new_multi(
    pair_deltas, panel, OUT_DIR / 'two_body_examples_existing_single_new_multi.png'
)
plot_pair_pattern_existing_single_new_multi(
    pair_deltas, panel, OUT_DIR / 'two_body_pattern_existing_single_new_multi.png'
)

display(Markdown('**Two-body plots**'))
display(Image(filename=str(OUT_DIR / 'two_body_summary_ranges.png')))
display(Image(filename=str(OUT_DIR / 'two_body_event_time_total.png')))
display(Image(filename=str(OUT_DIR / 'two_body_distance_bands_total.png')))
display(Image(filename=str(OUT_DIR / 'two_body_examples_total.png')))
display(Image(filename=str(OUT_DIR / 'two_body_examples_all_sites.png')))
display(Image(filename=str(OUT_DIR / 'two_body_examples_existing_single_new_multi.png')))
display(Image(filename=str(OUT_DIR / 'two_body_pattern_existing_single_new_multi.png')))


## 4. Three-body results

Now C is the newest launch month, and A/B are the two closest older sites within 10 miles that were already live at least 6 months earlier.


In [ ]:
triples_df = find_triples(sites, site_distances, max_neighbor_miles=MAX_NEIGHBOR_MILES, pre_buffer_months=PRE_POST_WINDOW)
triple_deltas = build_triple_deltas(panel, triples_df, pre_post_window=PRE_POST_WINDOW, min_months=3)
triple_summary = summarize_triples(triple_deltas)
triple_profile = build_triple_event_profile(triple_deltas, panel, window=PRE_POST_WINDOW)
triple_examples = choose_triple_examples(triple_deltas, n_examples=4)

triple_deltas.to_csv(OUT_DIR / 'three_body_triple_deltas.csv', index=False)
triple_summary.to_csv(OUT_DIR / 'three_body_summary.csv', index=False)
triple_profile.to_csv(OUT_DIR / 'three_body_event_profile.csv', index=False)

print(f"Usable three-body triples after the coverage filter: {len(triple_deltas)}")
display(Markdown(
    f"**Headline read**\n\n"
    f"- Median A-site total change after C launches: **{fmt_pct(triple_deltas['A_pct_wash_count_total'].median())}**\n"
    f"- Median B-site total change after C launches: **{fmt_pct(triple_deltas['B_pct_wash_count_total'].median())}**\n"
    f"- Median full A+B+C market total change: **{fmt_pct(triple_deltas['region_pct_wash_count_total'].median())}**"
))

triple_summary


In [ ]:
plot_triple_summary(triple_deltas, OUT_DIR / 'three_body_summary_ranges.png')
plot_triple_event_profile(triple_profile, OUT_DIR / 'three_body_event_time_total.png')
plot_triple_networks(triple_examples, OUT_DIR / 'three_body_networks.png')
plot_triple_examples_all(triple_deltas, panel, OUT_DIR / 'three_body_examples_all_sites.png')

display(Markdown('**Three-body plots**'))
display(Image(filename=str(OUT_DIR / 'three_body_summary_ranges.png')))
display(Image(filename=str(OUT_DIR / 'three_body_event_time_total.png')))
display(Image(filename=str(OUT_DIR / 'three_body_networks.png')))
display(Image(filename=str(OUT_DIR / 'three_body_examples_all_sites.png')))


## 5. Write the short report and keep only the presentation-ready artefacts


In [ ]:
report_path = write_report(
    OUT_DIR,
    panel,
    validation,
    pair_deltas,
    triple_deltas,
    pair_profile,
    triple_profile,
    max_neighbor_miles=MAX_NEIGHBOR_MILES,
    pre_post_window=PRE_POST_WINDOW,
)

keep = {
    'site_interaction_report.md',
    'two_body_pair_deltas.csv',
    'two_body_summary.csv',
    'two_body_event_profile.csv',
    'two_body_summary_ranges.png',
    'two_body_event_time_total.png',
    'two_body_distance_bands_total.png',
    'two_body_examples_total.png',
    'two_body_examples_all_sites.png',
    'two_body_examples_existing_single_new_multi.png',
    'two_body_pattern_existing_single_new_multi.png',
    'three_body_triple_deltas.csv',
    'three_body_summary.csv',
    'three_body_event_profile.csv',
    'three_body_summary_ranges.png',
    'three_body_event_time_total.png',
    'three_body_networks.png',
    'three_body_examples_all_sites.png',
}
curate_outputs(OUT_DIR, keep)

print('Markdown report written to', report_path)
display(Markdown(report_path.read_text()))
